In [4]:
import pandas as pd
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite
from camara_deputados.modeling.modeling import DataModeling
from camara_deputados.transformer.transformer import DataTransformer


In [5]:
# instâncias

silver = DataLoader('silver')
gold = DataLoader('gold')
salva= DataWrite()
transformer = DataTransformer()
modeling = DataModeling(transformer)


# Criando Dim na gold

In [3]:
dfs_silver = silver.carregar_parquets()


📂 Lendo dados da camada silver: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/data/silver



FileNotFoundError: ❌ Caminho não encontrado: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/data/silver

## Criando a dimensão dos deputados

In [55]:
df_s_deputados = dfs_silver['silver_deputado']



In [5]:
colunas_deputado = list(df_s_deputados.columns)
print(colunas_deputado)

['id_Mandato', 'nom_NomeCivil', 'nom_Sexo', 'dat_DataNasc', 'dat_DataFalecimento', 'nom_UFNasc', 'nom_MunicipioNasci', 'nom_Escolaridade', 'nom_SiglaPartido', 'nom_UFRepresenta', 'id_legislatura', 'nom_Email', 'nom_NomeEleitoral', 'nom_Situacao', 'nom_CondEleitoral', 'id_Deputado', 'data_extracao']


In [ ]:
colunas_g_dim_dep = [#'id_mandato'
 'nom_NomeCivil'
, 'nom_Sexo'
, 'dat_DataNasc'
, 'dat_DataFalecimento'
, 'nom_UFNasc'
, 'nom_MunicipioNasci'
#, 'nom_Escolaridade'
#, 'nom_SiglaPartido'
#, 'nom_UFRepresenta'
#, 'id_Legislatura'
##, 'nom_NomeEleitoral'
#, 'nom_Situacao'
#, 'nom_CondEleitoral'
#, 'data_extracao'
, 'id_Deputado'
]


In [58]:
dim_deputado = df_s_deputados[colunas_g_dim_dep].drop_duplicates(subset='id_Deputado')

In [61]:
dim_deputado.dtypes

nom_NomeCivil                  string
nom_Sexo                       string
dat_DataNasc           datetime64[us]
dat_DataFalecimento    datetime64[us]
nom_UFNasc                     string
nom_MunicipioNasci             string
id_Deputado                     Int64
dtype: object

In [13]:
salva.save_parquet(dim_deputado, 'dim_deputado', 'gold')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_deputado/dim_deputado.parquet


## Dim partidos

In [10]:
df_g_partido = modeling.criar_dim(
    df_s_deputados,
    colunas=['nom_SiglaPartido'],
    gerar_id=True,
    colunas_id=['nom_SiglaPartido'],
    nome_id='id_Partido'
)


In [11]:
df_g_partido.head()

,nom_SiglaPartido,id_Partido
0,DEM,377c92f580b6bdead551ea50221de7eb
6,PL,9b7d173b068dc4d5517bfae92d676437
7,PSC,79ebd5d2c77fb9a2a47cd341d84f96a4
11,UNIÃO,3dde549cf4ef6ec5e8112f1e12539dd6
13,MDB,6c152faa4bbca9a007fc7c608b6583a5


In [16]:
salva.save_parquet(
    df=df_g_partido,
    dataset='dim_partido',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_partido/dim_partido.parquet


## Criando a dim mandato 

Ao levantar o detalhamento dos deputados, observado que as linhas representam mandatos dos deputados

In [19]:
coluna_mandato = ['id_Mandato'
, 'nom_SiglaPartido'
, 'nom_UFRepresenta'
, 'id_legislatura'
, 'id_Deputado'
]

df_dim_mandato = df_s_deputados[coluna_mandato].drop_duplicates(subset='id_Mandato')

In [20]:
print(df_dim_mandato.head())
print(df_dim_mandato.columns)
df_dim_mandato.info()

    id_Mandato nom_SiglaPartido nom_UFRepresenta  id_legislatura  id_Deputado
0       178957              DEM               RR              55       178957
6       220593               PL               MT              57       220593
7       204554              PSC               BA              56       204554
11      204521            UNIÃO               SP              56       204521
13      204379              MDB               AP              57       204379
Index(['id_Mandato', 'nom_SiglaPartido', 'nom_UFRepresenta', 'id_legislatura',
       'id_Deputado'],
      dtype='str')
<class 'pandas.DataFrame'>
Index: 395 entries, 0 to 996
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_Mandato        395 non-null    Int64 
 1   nom_SiglaPartido  395 non-null    string
 2   nom_UFRepresenta  395 non-null    string
 3   id_legislatura    395 non-null    Int64 
 4   id_Deputado       395 non-null    Int64 
d

In [21]:
#inclui o id do partido

df_mandato_final = df_dim_mandato.merge(
    df_g_partido,
    on="nom_SiglaPartido",
    how="left"
)

In [23]:
dim_mandato = modeling.criar_dim(
    df=df_mandato_final,
    colunas=[
        "id_Mandato",
        "id_Deputado",
        "id_Partido",
        "id_legislatura",
        "nom_UFRepresenta"
    ],
    chave_duplicidade=["id_Mandato"]
)

In [24]:
dim_mandato.head()

,id_Mandato,id_Deputado,id_Partido,id_legislatura,nom_UFRepresenta
0,178957,178957,377c92f580b6bdead551ea50221de7eb,55,RR
1,220593,220593,9b7d173b068dc4d5517bfae92d676437,57,MT
2,204554,204554,79ebd5d2c77fb9a2a47cd341d84f96a4,56,BA
3,204521,204521,3dde549cf4ef6ec5e8112f1e12539dd6,56,SP
4,204379,204379,6c152faa4bbca9a007fc7c608b6583a5,57,AP


In [25]:
salva.save_parquet(
    df=dim_mandato,
    dataset='dim_mandato',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_mandato/dim_mandato.parquet


## DIM Proposição


In [26]:
df_silver_proposicao = dfs_silver['silver_proposica']

In [ ]:
dim_proposicao = modeling.criar_dim(
    df=df_silver_proposicao,
    colunas=[
        "id_proposicao",
        "cod_Tipo",
        "nom_TipoProposicao",
        "num_NumeroProp",
        "num_ano",
        "nom_Ementa",
        "nom_regime",
        "dat_Apresentacao"
    ],
    chave_duplicidade=["id_proposicao"]
)

In [30]:
salva.save_parquet(
    df=dim_proposicao,
    dataset='dim_proposicao',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_proposicao/dim_proposicao.parquet


## Criando uma dim_temas

In [31]:
df_temas = transformer.split_and_explode(
    df_silver_proposicao,
    coluna="nom_Keywords",
    nova_coluna="tema"
)

In [32]:
df_temas.head()

,id_proposicao,nom_TipoProposicao,cod_Tipo,num_NumeroProp,num_ano,nom_Ementa,dat_Apresentacao,uri_Autor,nom_Keywords,nom_SiglaOrgao,...,cod_TipoTramitacao,cod_TipoSiituacao,cod_Situacao,nom_LinkProposicao,nom_TipoAutor,id_Autor,nom_TipoRelator,id_Relator,data_extracao,tema
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,obrigatoriedade
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,homologação
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,cartório
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,reconhecimento de firma
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,crédito consignado


In [33]:
dim_tema = modeling.criar_dim(
    df=df_temas,
    colunas=["tema"],
    gerar_id=True,
    colunas_id=["tema"],
    nome_id="id_tema"
)

In [51]:
dim_tema.head()

NameError: name 'dim_tema' is not defined

# Bridge proposição tema

In [50]:
bridge_proposicao_tema = (
    df_temas
    .merge(dim_tema, on="tema", how="left")
    [["id_proposicao", "id_tema"]]
    .drop_duplicates()
)

NameError: name 'df_temas' is not defined

In [49]:
bridge_proposica_tema


KeyboardInterrupt



salvando na gold

In [36]:
salva.save_parquet(dim_proposicao, "dim_proposicao", "gold")
salva.save_parquet(dim_tema, "dim_tema", "gold")
salva.save_parquet(bridge_proposicao_tema, "bridge_proposicao_tema", "gold")

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_proposicao/dim_proposicao.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_tema/dim_tema.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/bridge_proposicao_tema/bridge_proposicao_tema.parquet


## Dim_Autor

In [37]:
df_silver_autores = dfs_silver['silver_autores']

In [38]:
df_silver_autores.columns

Index(['uri_Autor', 'nom_Autor', 'cod_TipoAutor', 'nom_TipoAutor',
       'num_OrdemAssinatura', 'ind_Proponente', 'id_Proposicao',
       'dat_Extracao', 'id_Autor', 'data_extracao'],
      dtype='str')

In [39]:
df_silver_autores.head()

,uri_Autor,nom_Autor,cod_TipoAutor,nom_TipoAutor,num_OrdemAssinatura,ind_Proponente,id_Proposicao,dat_Extracao,id_Autor,data_extracao
0,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2167556,2026-05-01 14:19:06.088189,253,2026-05-01 14:51:14.844471
1,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2167557,2026-05-01 14:19:06.088189,253,2026-05-01 14:51:14.844471
2,https://dadosabertos.camara.leg.br/api/v2/depu...,Rubens Bueno,10000,deputados,1,1,2167558,2026-05-01 14:19:06.088189,73466,2026-05-01 14:51:14.844471
3,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2167559,2026-05-01 14:19:06.088189,253,2026-05-01 14:51:14.844471
4,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2167562,2026-05-01 14:19:06.088189,253,2026-05-01 14:51:14.844471


In [40]:
df_silver_autores[['cod_TipoAutor','nom_TipoAutor','nom_Autor']].drop_duplicates()

,cod_TipoAutor,nom_TipoAutor,nom_Autor
0,30000,orgaos,Poder Executivo
2,10000,deputados,Rubens Bueno
5,10000,deputados,Reginaldo Lopes
6,10000,deputados,Arnaldo Jordy
8,10000,deputados,Antonio Carlos Mendes Thame
...,...,...,...
305,81001,orgaos,CONGRESSO NACIONAL
314,50000,orgaos,Superior Tribunal de Justiça
319,2,orgaos,Comissão de Constituição e Justiça e de Cidadania
330,80000,orgaos,Liderança do Partido Socialista Brasileiro


In [41]:
df_silver_autores.info()


<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   uri_Autor            343 non-null    string        
 1   nom_Autor            343 non-null    string        
 2   cod_TipoAutor        343 non-null    Int64         
 3   nom_TipoAutor        343 non-null    str           
 4   num_OrdemAssinatura  343 non-null    Int64         
 5   ind_Proponente       343 non-null    string        
 6   id_Proposicao        343 non-null    Int64         
 7   dat_Extracao         343 non-null    datetime64[us]
 8   id_Autor             343 non-null    Int64         
 9   data_extracao        343 non-null    datetime64[us]
dtypes: Int64(4), datetime64[us](2), str(1), string(3)
memory usage: 55.5 KB


In [42]:
dim_autor = modeling.criar_dim(
    df=df_silver_autores,
    colunas=[
        "id_Autor",
        'nom_Autor',
        "cod_TipoAutor"
    ],
    chave_duplicidade=["id_Autor"]
)

In [45]:
dim_autor["id_Autor"] = dim_autor["id_Autor"].astype("Int64")
dim_deputado["id_Deputado"] = dim_deputado["id_Deputado"].astype("Int64")

In [48]:
df_validacao = dim_autor.merge(
    dim_deputado[["id_Deputado"]],
    left_on="id_Autor",
    right_on="id_Deputado",
    how="left",
    indicator=True
)


In [85]:
df_validacao.head(100)

,id_Autor,nom_Autor,cod_TipoAutor,id_deputado,_merge
0,141416,Edgar Moury,10000,<NA>,left_only
1,160673,Giovani Cherini,10000,160673,both
2,178930,Evandro Rogerio Roman,10000,<NA>,left_only
3,141422,Efraim Filho,10000,<NA>,left_only
4,160565,Mara Gabrilli,10000,<NA>,left_only
5,133439,André Figueiredo,10000,133439,both
6,253,Poder Executivo,30000,<NA>,left_only
7,78,Senado Federal - Roberto Muniz,40000,<NA>,left_only
8,74274,José Eduardo Cardozo,10000,<NA>,left_only
9,74665,Jaime Martins,10000,<NA>,left_only


## Dim Tipo Autor

In [49]:
df_dim_tipoAutor = dfs_silver['silver_tipos_autores']

In [50]:
df_dim_tipoAutor.columns

Index(['id_Autor', 'nom_NomeAutor', 'data_extracao'], dtype='str')

In [51]:
dim_tipoAutor = modeling.criar_dim(
    df_dim_tipoAutor,
    ['id_Autor', 'nom_NomeAutor'],
     chave_duplicidade=["id_Autor"]
)

## bridge proposicao autor

In [53]:
bridge_proposicao_autor = (
    df_silver_autores[["id_Proposicao", "id_Autor"]]
    .drop_duplicates()
)

In [54]:
salva.save_parquet(dim_autor, "dim_autor", "gold")
salva.save_parquet(bridge_proposicao_autor, "bridge_proposicao_autor", "gold")

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_autor/dim_autor.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/bridge_proposicao_autor/bridge_proposicao_autor.parquet


## Votações

In [69]:
df_votacoes_detalhamento = dfs_silver['silver_votacao_detalhamento']

In [70]:
df_votacoes_proposicao= dfs_silver['silver_votacao_proposicao']

In [71]:
df_votacoes_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 819 entries, 0 to 818
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_Votacao        819 non-null    string        
 1   dat_DataVotacao   819 non-null    datetime64[us]
 2   dat_DataRegistro  819 non-null    datetime64[us]
 3   uri_Orgao         819 non-null    string        
 4   nom_Descricao     819 non-null    string        
 5   ind_Aprovado      797 non-null    string        
 6   id_Proposicao     819 non-null    Int64         
 7   id_Orgao          819 non-null    Int64         
 8   data_extracao     819 non-null    datetime64[us]
dtypes: Int64(2), datetime64[us](3), string(4)
memory usage: 164.2 KB


In [65]:
df_votacoes_detalhamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 819 entries, 0 to 818
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id_Votacao                  819 non-null    string        
 1   dat_DataVotacao             819 non-null    datetime64[us]
 2   dat_DataHoraRegistro        819 non-null    datetime64[us]
 3   nom_SiglaOrgao              819 non-null    string        
 4   id_Orgao                    819 non-null    Int64         
 5   id_Evento                   732 non-null    Int64         
 6   des_DescricaoVotacao        819 non-null    string        
 7   ind_Aprovacao               797 non-null    string        
 8   des_UltimaAbertura          193 non-null    string        
 9   dat_UltimaAbertura          193 non-null    datetime64[us]
 10  des_Efeitos                 819 non-null    string        
 11  des_Objetos                 819 non-null    string        
 12  des_P

In [74]:
df_votacoes_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 819 entries, 0 to 818
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_Votacao        819 non-null    string        
 1   dat_DataVotacao   819 non-null    datetime64[us]
 2   dat_DataRegistro  819 non-null    datetime64[us]
 3   uri_Orgao         819 non-null    string        
 4   nom_Descricao     819 non-null    string        
 5   ind_Aprovado      797 non-null    string        
 6   id_Proposicao     819 non-null    Int64         
 7   id_Orgao          819 non-null    Int64         
 8   data_extracao     819 non-null    datetime64[us]
dtypes: Int64(2), datetime64[us](3), string(4)
memory usage: 164.2 KB


In [73]:
df_votacoes_detalhamento.columns

Index(['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao',
       'idOrgao', 'uriEvento', 'idEvento', 'descricao', 'aprovacao',
       'descUltimaAberturaVotacao', 'dataHoraUltimaAberturaVotacao',
       'efeitosRegistrados', 'objetosPossiveis',
       'ultimaApresentacaoProposicao.dataHoraRegistro',
       'ultimaApresentacaoProposicao.descricao',
       'ultimaApresentacaoProposicao.uriProposicaoCitada', 'source_id',
       'data_extracao', 'id_Proposicao', 'nom_SiglaTipoProposicao',
       'num_Proposicao', 'num_AnoProposicao'],
      dtype='str')

In [75]:
df_votacoes_proposicao.head()

,id_Votacao,dat_DataVotacao,dat_DataRegistro,uri_Orgao,nom_Descricao,ind_Aprovado,id_Proposicao,id_Orgao,data_extracao
0,2167556-58,2018-05-23,2018-05-23 18:53:32,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Redação Final assinada pelo Relator...,1.0,2167556,180,2026-05-01 14:52:25.821904
1,2167556-56,2018-05-23,2018-05-23 18:52:29,https://dadosabertos.camara.leg.br/api/v2/orga...,"Aprovada a Medida Provisória nº 817 de 2018, n...",1.0,2167556,180,2026-05-01 14:52:25.821904
2,2167556-54,2018-05-23,2018-05-23 18:48:37,https://dadosabertos.camara.leg.br/api/v2/orga...,"Aprovado, em apreciação preliminar, o Parecer ...",1.0,2167556,180,2026-05-01 14:52:25.821904
3,2167559-66,2018-05-23,2018-05-23 20:50:02,https://dadosabertos.camara.leg.br/api/v2/orga...,Suprimido o texto.,<NA>,2167559,180,2026-05-01 14:52:25.821904
4,2167559-60,2018-05-23,2018-05-23 20:33:46,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Redação Final assinada pelo Relator...,1.0,2167559,180,2026-05-01 14:52:25.821904


In [76]:
df_votacoes_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 819 entries, 0 to 818
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_Votacao        819 non-null    string        
 1   dat_DataVotacao   819 non-null    datetime64[us]
 2   dat_DataRegistro  819 non-null    datetime64[us]
 3   uri_Orgao         819 non-null    string        
 4   nom_Descricao     819 non-null    string        
 5   ind_Aprovado      797 non-null    string        
 6   id_Proposicao     819 non-null    Int64         
 7   id_Orgao          819 non-null    Int64         
 8   data_extracao     819 non-null    datetime64[us]
dtypes: Int64(2), datetime64[us](3), string(4)
memory usage: 164.2 KB


In [77]:
df_votacoes_proposicao.columns

Index(['id_Votacao', 'dat_DataVotacao', 'dat_DataRegistro', 'uri_Orgao',
       'nom_Descricao', 'ind_Aprovado', 'id_Proposicao', 'id_Orgao',
       'data_extracao'],
      dtype='str')

## Fato_Votacao

In [87]:
mapping_votacao ={
    "id_Votacao": ("id_Votacao", "str"),
    "id_Orgao": ("id_Orgao", "int"),
    "dat_DataVotacao": ("dat_Votacao", "date"),
    "dat_DataRegistro": ("dat_Registro", "date"),
    "ind_Aprovado": ("ind_Aprovado", "int"),
}

df_fatovotacao = transformer.rename_and_cast(
    df_votacoes_proposicao,
    mapping_votacao

)

df_fatovotacao = df_fatovotacao.drop_duplicates(subset=["id_Votacao"])

In [88]:
df_fatovotacao["id_Votacao"].is_unique

True

In [89]:
salva.save_parquet(
    df_fatovotacao,
    'fato_votacao',
    'gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/fato_votacao/fato_votacao.parquet


## ponte votacao preposicao

In [96]:
df_bridge = df_votacoes_proposicao[[
    "id_Votacao",
    "id_Proposicao"
]].copy()

mapping = {
    "id_votacao": ("id_Votacao", "str"),
    "id_Proposicao": ("id_Proposicao", "int"),
}


df_bridge = transformer.rename_and_cast(df_bridge, mapping)

df_bridge = df_bridge.drop_duplicates()

In [100]:
salva.save_parquet(df_bridge, "bridge_votacao_proposicao", "gold")


💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/bridge_votacao_proposicao/bridge_votacao_proposicao.parquet


## fato voto deputado

In [105]:
df_voto_deputado = dfs_silver['silver_votos_deputados']

In [107]:
df_fato_voto = df_voto_deputado[[
    "id_Votacao",
    "id_Deputado",
    "nom_Voto"
]].copy()

In [110]:
df_fato_voto = df_fato_voto.drop_duplicates(
    subset=["id_Votacao", "id_Deputado"])

In [ ]:
salva.save_parquet(df_fato_voto,
'fato_voto',
'gold')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/fato_voto/fato_voto.parquet


## fato orientaçao partido

In [36]:
df_orientacao = dfs_silver['silver_orientacao']

In [37]:
df_orientacao.head()

,nom_OrientacaoVoto,id_TipoLideranca,nom_SiglaPartidoBloco,id_PartidoBloco,des_UriPartidoBloco,id_Votacao,dat_DataExtracao,data_extracao
0,Sim,<NA>,NOVO,37901,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038,2026-05-01 20:28:00.094013
1,,<NA>,PSB,36832,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038,2026-05-01 20:28:00.094013
2,Sim,<NA>,Oposição,<NA>,<NA>,559138-241,2026-05-01 14:46:37.688038,2026-05-01 20:28:00.094013
3,Não,<NA>,Governo,<NA>,<NA>,559138-241,2026-05-01 14:46:37.688038,2026-05-01 20:28:00.094013
4,Sim,<NA>,PL,37906,https://dadosabertos.camara.leg.br/api/v2/part...,559138-241,2026-05-01 14:46:37.688038,2026-05-01 20:28:00.094013


In [39]:
mapping = {
    "id_Votacao": ("id_Votacao", "str"),
    "nom_SiglaPartidoBloco": ("nom_SiglaPartido", "str"),
    "nom_OrientacaoVoto": ("nom_Orientacao", "str"),
}

In [40]:
df_orientacao = transformer.rename_and_cast(df_orientacao, mapping)

In [41]:
df_orientacao.head()

,id_Votacao,nom_SiglaPartido,nom_Orientacao
0,559138-241,NOVO,Sim
1,559138-241,PSB,
2,559138-241,Oposição,Sim
3,559138-241,Governo,Não
4,559138-241,PL,Sim


In [42]:
df_orientacao = df_orientacao.merge(
    df_g_partido,
    on="nom_SiglaPartido",
    how="left"
)

In [43]:
df_orientacao = df_orientacao.drop(columns=["nom_SiglaPartido"])

In [45]:
salva.save_parquet(df_orientacao, "fato_orientacao_partido", "gold")

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/fato_orientacao_partido/fato_orientacao_partido.parquet


In [46]:
def criar_dim_tempo(data_inicio, data_fim):

    datas = pd.date_range(start=data_inicio, end=data_fim)

    df = pd.DataFrame({
        "data": datas
    })

    df["ano"] = df["data"].dt.year
    df["mes"] = df["data"].dt.month
    df["dia"] = df["data"].dt.day
    df["trimestre"] = df["data"].dt.quarter

    return df

In [47]:
df_tempo = criar_dim_tempo("2000-01-01", "2030-12-31")


In [48]:
salva.save_parquet(
    df_tempo,
    'dim_Periodo',
    'gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_Periodo/dim_Periodo.parquet
